# 8.2 Direct scoring: `score(image, configuration)`

This notebook calls Python functions directly. It needs **no BrandLens FastAPI server, database, upload route, or job worker**. The two inputs are a loaded Pillow image and the tuple returned by `load_cv_config()`.

A JSON selecting VLM criteria still needs a running vision-model endpoint. The existing `ModelClient` sends the photograph there; importing it from `brandlens.api` does not start or call the BrandLens API.

The default three-criterion test JSON uses the unchanged VLM's supported tone list. The full test JSON includes all six criteria and intentionally leaves uncalibrated thresholds unavailable. The original brand JSON's tone-list mismatch remains explicit; tones are never substituted silently.

In [6]:
import json
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src" / "brandlens").is_dir()
    and (path / "config" / "brand_standards.json").is_file()
)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

In [7]:
from brandlens.api.model_client import ModelClient, ModelFailure
from brandlens.config import load_cv_config
from brandlens.cv.subject.rtdetr_detector import RTDetrSubjectDetector
from brandlens.cv.subject.selector import SingleSubjectSelector
from brandlens.pipeline import score_photo
from brandlens.preprocessing.image_io import load_image
from brandlens.preprocessing.standardization import StandardizationSettings

## Setup once

Set the image, JSON, reachable model URL, and model name actually served at that URL. The repository default model name below is not a guarantee that it is installed.

For **local Ollama on Windows**, start the Ollama application or run `ollama serve` in PowerShell. In another terminal, `ollama list` shows installed model names. The default URL is `http://127.0.0.1:11434/v1`. If the model runs on another computer or inside a separate environment, use an address reachable from the notebook kernel.

For subject measurements, select the full test JSON, install the existing optional subject dependencies, and enable the detector. No detector is needed for the default three criteria.

In [8]:
IMAGE_PATH = ROOT / "vlm" / "samples" / "01-coffee-shop.jpg"
CONFIG_PATH = ROOT / "config" / "brand_standards_cv_vlm_test.json"
# CONFIG_PATH = ROOT / "config" / "brand_standards_full_pipeline_test.json"
SCHEMA_PATH = ROOT / "config" / "brand_standards.schema.json"

MODEL_BASE_URL = os.getenv("BRANDLENS_MODEL_BASE_URL", "http://127.0.0.1:11434/v1")
# Repository default; change this to the vision model actually served by your endpoint.

MODEL_NAME = os.getenv("BRANDLENS_MODEL_NAME", "gemma4:e4b")
MODEL_NAME = "qwen3.5:0.8b" #SMALLER MODEL FOR LOCAL RUN

MODEL_API_KEY = os.getenv("BRANDLENS_MODEL_API_KEY") or None
MODEL_TIMEOUT_S = float(os.getenv("BRANDLENS_MODEL_TIMEOUT_S", "60"))
MODEL_MAX_RETRIES = int(os.getenv("BRANDLENS_MODEL_MAX_RETRIES", "1"))
RUN_SUBJECT_DETECTOR = False
SHOW_MEASUREMENTS = True

configuration = load_cv_config(CONFIG_PATH, schema_path=SCHEMA_PATH)
standardization_settings = StandardizationSettings(max_size=(2048, 2048))
needs_subject = any(
    criterion.input_requirements().get("reliable_subject_region", False)
    for criterion in configuration[1].values()
)
subject_detector = RTDetrSubjectDetector() if needs_subject and RUN_SUBJECT_DETECTOR else None
subject_selector = SingleSubjectSelector() if subject_detector is not None else None

In [9]:
def score(image, configuration):
    """Score one loaded image using the initialized configuration.

    Returns the existing (ImageScore, CV results, VLM results) tuple.
    This notebook convenience wrapper opens/closes the HTTP client for each
    call; the vision model remains hosted by the existing model endpoint.
    For batches, keep one client open and call score_photo directly.
    """
    try:
        with ModelClient(
            MODEL_BASE_URL,
            MODEL_NAME,
            api_key=MODEL_API_KEY,
            timeout_s=MODEL_TIMEOUT_S,
            max_retries=MODEL_MAX_RETRIES,
        ) as model:
            return score_photo(
                image,
                configuration,
                model,
                subject_detector=subject_detector,
                subject_selector=subject_selector,
                standardization_settings=standardization_settings,
            )
    except ModelFailure as error:
        print("VLM failure reason:", error.reason)
        print("Details:", error.message)
        print("Attempts:", error.attempts)
        if error.reason == "connection":
            print("Check that the model endpoint is running and reachable from this notebook kernel.")
        raise

In [10]:
# a = loaded Pillow image; b = configuration already loaded above.
# No FastAPI server, API job, database, or worker is involved.
scored, cv_results, vlm_results = None, {}, {}
with load_image(IMAGE_PATH) as image:
    scored, cv_results, vlm_results = score(image, configuration)

In [11]:
if scored is None:
    raise RuntimeError("No score was produced. Resolve the preceding VLM error first.")

overall = pd.DataFrame([{
    "rubric_id": scored.rubric_id,
    "overall_score": scored.overall_score,
    "status": scored.status,
    "scored_criteria": sum(row.availability == "ok" for row in scored.criteria),
    "configured_criteria": len(scored.criteria),
}])
print("Overall score (0–100)")
display(overall)

individual = pd.DataFrame([{
    "criterion_id": row.criterion_id,
    "source": row.source,
    "score": row.score,
    "applied_weight": row.weight,
    "availability": row.availability,
    "reason": "; ".join(row.reasons),
} for row in scored.criteria])
print("Individual scores (0–100)")
display(individual)

Overall score (0–100)


,rubric_id,overall_score,status,scored_criteria,configured_criteria
0,cv-vlm-integration-test,100.0,scored,3,3


Individual scores (0–100)


,criterion_id,source,score,applied_weight,availability,reason
0,exposure_brightness,cv,100.0,0.333333,ok,
1,contrast,cv,100.0,0.333333,ok,
2,tone_approachability,vlm,100.0,0.333333,ok,


In [12]:
if SHOW_MEASUREMENTS and scored is not None:
    measurement_rows = [{
        "criterion_id": cid,
        "measurement": name,
        "value": json.dumps(value) if isinstance(value, (list, dict)) else value,
        "unit": result["units"].get(name, ""),
    } for cid, result in cv_results.items()
      for name, value in result["measurements"].items()]
    measurements = pd.DataFrame(
        measurement_rows, columns=["criterion_id", "measurement", "value", "unit"]
    )
    
    print("CV measurements:")
    display(measurements)
    
    judgments = pd.DataFrame([{
        "criterion_id": cid,
        "model_id": result["id"],
        "tone_label": result["tone_label"],
        "fit_score_1_to_5": result["fit_score"],
        "evidence": result["evidence"],
    } for cid, result in vlm_results.items()])
    
    print("VLM judgment:")
    display(judgments)

CV measurements:


,criterion_id,measurement,value,unit
0,exposure_brightness,mean_luminance,95.686399,grayscale_0_255
1,exposure_brightness,near_black_fraction,0.000000,fraction_0_1
2,exposure_brightness,near_white_fraction,0.005723,fraction_0_1
3,contrast,luminance_std,68.646647,grayscale_0_255
4,contrast,p5_p95_range,194.000000,grayscale_0_255
5,contrast,near_black_fraction,0.000000,fraction_0_1
6,contrast,near_white_fraction,0.005723,fraction_0_1


VLM judgment:


,criterion_id,model_id,tone_label,fit_score_1_to_5,evidence
0,tone_approachability,B1,"authentic, casual, genuine, family, premium, m...",5,The photograph captures a candid moment of a b...
